# Building Production RAG Systems

## Complete Hands-On Guide to Retrieval-Augmented Generation

In this notebook, you will:
1. Understand RAG architecture
2. Build a complete RAG pipeline from scratch
3. Implement hybrid search (dense + sparse)
4. Add reranking for better results
5. Evaluate RAG performance

**Prerequisites:**
- Python 3.10+
- Basic knowledge of embeddings and vector search

**Estimated Time:** 2-3 hours

## Step 1: Environment Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu
!pip install transformers torch
!pip install numpy pandas

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from dataclasses import dataclass
import hashlib

print("✓ Libraries loaded successfully")

## Step 2: RAG Architecture Overview

In [ ]:
from IPython.display import Markdown

rag_architecture = """
## Complete RAG Pipeline

```
┌─────────────────────────────────────────────────────────┐
│                    RAG Pipeline                          │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  Document → Chunking → Embedding → Vector Store         │
│                                            ↓             │
│  User Query → Embedding → Retrieval → Reranking         │
│                                            ↓             │
│                                    Build Context         │
│                                            ↓             │
│                                    LLM Generation        │
│                                            ↓             │
│                                    Answer + Sources      │
└─────────────────────────────────────────────────────────┘
```

### Key Components:
1. **Document Ingestion**: Load and preprocess documents
2. **Chunking**: Split into manageable pieces
3. **Embedding**: Convert to vector representations
4. **Retrieval**: Find relevant chunks (hybrid search)
5. **Reranking**: Improve relevance with cross-encoder
6. **Generation**: LLM produces answer with context
"""

Markdown(rag_architecture)

## Step 3: Create Sample Documents

In [ ]:
# Sample knowledge base
documents = [
    {
        "id": "doc1",
        "title": "Introduction to Machine Learning",
        "content": """
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that can access data and use it to learn for themselves. 
The primary aim is to allow computers to learn automatically without human intervention or assistance.
Key types include supervised learning, unsupervised learning, and reinforcement learning.
"""
    },
    {
        "id": "doc2",
        "title": "Deep Learning Fundamentals",
        "content": """
Deep learning is a subset of machine learning that uses neural networks with multiple layers (deep neural networks).
These networks learn hierarchical representations of data, from simple features to complex abstractions.
Common architectures include Convolutional Neural Networks (CNNs) for images and Recurrent Neural Networks (RNNs) for sequences.
Transformers, introduced in 2017, have revolutionized natural language processing.
"""
    },
    {
        "id": "doc3",
        "title": "Natural Language Processing",
        "content": """
Natural Language Processing (NLP) is a field of AI focused on enabling computers to understand, interpret, and generate human language.
Key tasks include text classification, named entity recognition, sentiment analysis, and machine translation.
Modern NLP relies heavily on transformer models like BERT, GPT, and their variants.
Applications include chatbots, virtual assistants, and automated content generation.
"""
    },
    {
        "id": "doc4",
        "title": "Computer Vision Basics",
        "content": """
Computer vision enables machines to interpret and understand visual information from the world.
Applications include image classification, object detection, facial recognition, and autonomous vehicles.
Convolutional Neural Networks (CNNs) are the backbone of modern computer vision systems.
Recent advances include vision transformers (ViT) that apply transformer architecture to images.
"""
    },
    {
        "id": "doc5",
        "title": "Reinforcement Learning",
        "content": """
Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by interacting with an environment.
The agent receives rewards or penalties based on its actions and learns to maximize cumulative reward.
Key concepts include Q-learning, policy gradients, and actor-critic methods.
Applications include game playing (AlphaGo), robotics, and resource optimization.
"""
    }
]

print(f"Created {len(documents)} sample documents")

## Step 4: Implement Chunking Strategy

In [ ]:
@dataclass
class Chunk:
    """Document chunk with metadata"""
    id: str
    text: str
    document_id: str
    document_title: str
    chunk_index: int
    metadata: Dict
    embedding: np.ndarray = None

class TextChunker:
    """Split documents into overlapping chunks"""
    
    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def chunk_documents(self, documents: List[Dict]) -> List[Chunk]:
        """Split documents into chunks"""
        all_chunks = []
        
        for doc in documents:
            doc_chunks = self._chunk_text(
                doc["content"],
                doc["id"],
                doc.get("title", "Untitled")
            )
            all_chunks.extend(doc_chunks)
        
        return all_chunks
    
    def _chunk_text(self, text: str, doc_id: str, title: str) -> List[Chunk]:
        """Split single document into chunks"""
        # Simple word-based chunking
        words = text.split()
        chunks = []
        
        for i in range(0, len(words), self.chunk_size - self.chunk_overlap):
            chunk_text = ' '.join(words[i:i + self.chunk_size])
            
            chunk = Chunk(
                id=f"{doc_id}_chunk_{len(chunks)}",
                text=chunk_text,
                document_id=doc_id,
                document_title=title,
                chunk_index=len(chunks),
                metadata={"doc_id": doc_id, "title": title}
            )
            chunks.append(chunk)
        
        return chunks

# Create chunks
chunker = TextChunker(chunk_size=200, chunk_overlap=50)
chunks = chunker.chunk_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nSample chunk:")
print(f"ID: {chunks[0].id}")
print(f"Text: {chunks[0].text[:100]}...")
print(f"Title: {chunks[0].document_title}")

## Step 5: Generate Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

class EmbeddingGenerator:
    """Generate embeddings for text"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        print("✓ Model loaded")
    
    def embed_chunks(self, chunks: List[Chunk]) -> List[Chunk]:
        """Generate embeddings for all chunks"""
        texts = [chunk.text for chunk in chunks]
        
        print(f"Generating embeddings for {len(texts)} chunks...")
        embeddings = self.model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        
        # Attach embeddings to chunks
        for chunk, embedding in zip(chunks, embeddings):
            chunk.embedding = embedding
        
        print(f"✓ Generated embeddings with shape: {embeddings.shape}")
        return chunks
    
    def embed_query(self, query: str) -> np.ndarray:
        """Generate embedding for query"""
        return self.model.encode(query, convert_to_numpy=True)

# Generate embeddings
embedder = EmbeddingGenerator(model_name="all-MiniLM-L6-v2")
chunks_with_embeddings = embedder.embed_chunks(chunks)

## Step 6: Build Hybrid Search (Dense + Sparse)

In [ ]:
import faiss
from rank_bm25 import BM25Okapi

class HybridSearch:
    """Hybrid search combining dense and sparse retrieval"""
    
    def __init__(self, chunks: List[Chunk]):
        self.chunks = chunks
        self._build_dense_index(chunks)
        self._build_sparse_index(chunks)
    
    def _build_dense_index(self, chunks: List[Chunk]):
        """Build FAISS index for dense retrieval"""
        embeddings = np.array([chunk.embedding for chunk in chunks])
        dimension = embeddings.shape[1]
        
        # Create FAISS index (Inner Product for cosine similarity)
        self.dense_index = faiss.IndexFlatIP(dimension)
        
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        self.dense_index.add(embeddings)
        
        print(f"✓ Built dense index with {self.dense_index.ntotal} vectors")
    
    def _build_sparse_index(self, chunks: List[Chunk]):
        """Build BM25 index for sparse retrieval"""
        # Tokenize documents
        tokenized_docs = [chunk.text.split() for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized_docs)
        print(f"✓ Built BM25 index with {len(tokenized_docs)} documents")
    
    def dense_search(self, query_embedding: np.ndarray, top_k: int = 5) -> List[Tuple[Chunk, float]]:
        """Search using dense embeddings"""
        # Normalize query
        query_norm = query_embedding.reshape(1, -1)
        faiss.normalize_L2(query_norm)
        
        # Search
        scores, indices = self.dense_index.search(query_norm, top_k)
        
        results = [(self.chunks[idx], float(score)) for idx, score in zip(indices[0], scores[0])]
        return results
    
    def sparse_search(self, query: str, top_k: int = 5) -> List[Tuple[Chunk, float]]:
        """Search using BM25 (keyword matching)"""
        query_tokens = query.split()
        scores = self.bm25.get_scores(query_tokens)
        
        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = [(self.chunks[idx], float(scores[idx])) for idx in top_indices]
        return results
    
    def hybrid_search(self, query: str, query_embedding: np.ndarray, 
                     top_k: int = 5, alpha: float = 0.5) -> List[Chunk]:
        """
        Combine dense and sparse search using Reciprocal Rank Fusion (RRF)
        
        Args:
            query: Search query text
            query_embedding: Query embedding
            top_k: Number of results to return
            alpha: Weight for dense search (0.5 = equal weight)
        """
        # Get results from both methods
        dense_results = self.dense_search(query_embedding, top_k=top_k * 2)
        sparse_results = self.sparse_search(query, top_k=top_k * 2)
        
        # Reciprocal Rank Fusion
        chunk_scores = {}
        
        # Score from dense retrieval
        for rank, (chunk, score) in enumerate(dense_results):
            chunk_scores[chunk.id] = chunk_scores.get(chunk.id, 0) + alpha * (1.0 / (rank + 1))
        
        # Score from sparse retrieval
        for rank, (chunk, score) in enumerate(sparse_results):
            chunk_scores[chunk.id] = chunk_scores.get(chunk.id, 0) + (1 - alpha) * (1.0 / (rank + 1))
        
        # Sort by fused score
        sorted_chunks = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Return top-k chunks
        return [self.chunks[int(chunk_id.split('_')[1])] for chunk_id, _ in sorted_chunks[:top_k]]

# Build hybrid search index
hybrid_search = HybridSearch(chunks_with_embeddings)
print("✓ Hybrid search index built")

## Step 7: Implement Reranking

In [ ]:
from sentence_transformers import CrossEncoder

class Reranker:
    """Rerank retrieved chunks using cross-encoder"""
    
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        print(f"Loading reranker: {model_name}")
        self.model = CrossEncoder(model_name)
        print("✓ Reranker loaded")
    
    def rerank(self, query: str, chunks: List[Chunk], top_k: int = 3) -> List[Chunk]:
        """
        Rerank chunks based on query-chunk relevance
        
        Args:
            query: Search query
            chunks: Retrieved chunks
            top_k: Number of top results to return
        """
        if not chunks:
            return []
        
        # Create query-chunk pairs
        pairs = [[query, chunk.text] for chunk in chunks]
        
        # Get relevance scores
        scores = self.model.predict(pairs)
        
        # Sort by score
        chunk_score_pairs = list(zip(chunks, scores))
        chunk_score_pairs.sort(key=lambda x: x[1], reverse=True)
        
        # Return top-k chunks
        return [chunk for chunk, _ in chunk_score_pairs[:top_k]]

# Initialize reranker
reranker = Reranker()
print("✓ Reranking system ready")

## Step 8: Build Complete RAG Pipeline

In [ ]:
class RAGPipeline:
    """Complete RAG pipeline"""
    
    def __init__(self, chunks: List[Chunk], embedder, hybrid_search, reranker):
        self.chunks = chunks
        self.embedder = embedder
        self.hybrid_search = hybrid_search
        self.reranker = reranker
        
        # RAG configuration
        self.retrieval_top_k = 10
        self.rerank_top_k = 3
    
    def query(self, question: str) -> Tuple[str, List[Chunk]]:
        """
        Process query through complete RAG pipeline
        
        Args:
            question: User question
        
        Returns:
            Tuple of (answer, retrieved_chunks)
        """
        print(f"\nQuery: {question}")
        print("-" * 60)
        
        # Step 1: Generate query embedding
        query_embedding = self.embedder.embed_query(question)
        
        # Step 2: Hybrid retrieval
        print("Step 1: Retrieving relevant chunks...")
        retrieved_chunks = self.hybrid_search.hybrid_search(
            question,
            query_embedding,
            top_k=self.retrieval_top_k
        )
        print(f"  Retrieved {len(retrieved_chunks)} chunks")
        
        # Step 3: Reranking
        print("Step 2: Reranking chunks...")
        reranked_chunks = self.reranker.rerank(
            question,
            retrieved_chunks,
            top_k=self.rerank_top_k
        )
        print(f"  Top {len(reranked_chunks)} chunks after reranking")
        
        # Step 4: Build context
        context = self._build_context(reranked_chunks)
        
        # Step 5: Generate answer (using simple template for now)
        answer = self._generate_answer(question, context)
        
        return answer, reranked_chunks
    
    def _build_context(self, chunks: List[Chunk]) -> str:
        """Build context from retrieved chunks"""
        context_parts = []
        
        for i, chunk in enumerate(chunks, 1):
            context_parts.append(
                f"[Source {i}: {chunk.document_title}]\n{chunk.text}"
            )
        
        return "\n\n".join(context_parts)
    
    def _generate_answer(self, question: str, context: str) -> str:
        """
        Generate answer using LLM
        In production, this would call your LLM of choice
        """
        # Simple template-based answer
        answer = f"""Based on the retrieved context:

{context}

Answer to "{question}":
The information above provides relevant context for answering your question.
In a production system, an LLM would generate a coherent response using this context.
"""
        return answer
    
    def display_results(self, question: str, answer: str, chunks: List[Chunk]):
        """Display RAG results in formatted way"""
        print("\n" + "=" * 60)
        print("RAG RESULTS")
        print("=" * 60)
        print(f"\nQuestion: {question}\n")
        print(f"Answer:\n{answer}")
        
        print("\nSources:")
        for i, chunk in enumerate(chunks, 1):
            print(f"\n[{i}] {chunk.document_title}")
            print(f"    Text: {chunk.text[:150]}...")
        print("=" * 60)

# Initialize RAG pipeline
rag_pipeline = RAGPipeline(
    chunks=chunks_with_embeddings,
    embedder=embedder,
    hybrid_search=hybrid_search,
    reranker=reranker
)

print("✓ RAG Pipeline initialized")

## Step 9: Test the RAG System

In [ ]:
# Test queries
test_queries = [
    "What is machine learning?",
    "How does deep learning work?",
    "What are neural networks?",
    "Explain reinforcement learning",
    "What is NLP used for?"
]

# Run tests
for query in test_queries:
    answer, chunks = rag_pipeline.query(query)
    rag_pipeline.display_results(query, answer, chunks)

## Step 10: Evaluate RAG Performance

In [ ]:
class RAGEvaluator:
    """Evaluate RAG system performance"""
    
    def __init__(self):
        self.test_cases = [
            {
                "query": "What is machine learning?",
                "expected_topics": ["machine learning", "artificial intelligence", "learn from data"]
            },
            {
                "query": "What are CNNs used for?",
                "expected_topics": ["computer vision", "image", "convolutional"]
            },
            {
                "query": "How does reinforcement learning work?",
                "expected_topics": ["reinforcement learning", "agent", "reward", "environment"]
            }
        ]
    
    def evaluate_retrieval(self, query: str, chunks: List[Chunk]) -> Dict:
        """Evaluate retrieval quality"""
        # Check if retrieved chunks contain expected topics
        test_case = next((tc for tc in self.test_cases if tc["query"] == query), None)
        
        if not test_case:
            return {"error": "No test case found"}
        
        # Check topic coverage
        chunk_text = " ".join([chunk.text.lower() for chunk in chunks])
        
        topics_found = sum(
            1 for topic in test_case["expected_topics"]
            if topic.lower() in chunk_text
        )
        
        coverage = topics_found / len(test_case["expected_topics"])
        
        return {
            "query": query,
            "topics_found": topics_found,
            "total_topics": len(test_case["expected_topics"]),
            "coverage": coverage,
            "num_chunks": len(chunks)
        }
    
    def run_evaluation(self, rag_pipeline: RAGPipeline) -> pd.DataFrame:
        """Run evaluation on all test cases"""
        results = []
        
        for test_case in self.test_cases:
            query = test_case["query"]
            answer, chunks = rag_pipeline.query(query)
            
            eval_result = self.evaluate_retrieval(query, chunks)
            results.append(eval_result)
        
        return pd.DataFrame(results)

# Run evaluation
evaluator = RAGEvaluator()
eval_results = evaluator.run_evaluation(rag_pipeline)

print("\nRAG Evaluation Results")
print("=" * 60)
print(eval_results)
print("\nAverage Coverage:", eval_results["coverage"].mean())

## Summary and Next Steps

### What You Built:
✓ Complete RAG pipeline from scratch
✓ Hybrid search (dense + sparse retrieval)
✓ Reranking with cross-encoder
✓ Evaluation framework

### Key Concepts:
1. **Chunking**: Split documents into overlapping segments
2. **Embeddings**: Convert text to vector representations
3. **Hybrid Search**: Combine semantic + keyword search
4. **Reranking**: Improve relevance with cross-encoders
5. **RRF Fusion**: Reciprocal Rank Fusion for combining results

### Next Steps:
1. **Add LLM Integration**: Connect to OpenAI, Anthropic, or local LLM
2. **Use Vector Database**: Qdrant, Pinecone, or Milvus for scale
3. **Advanced Chunking**: Semantic chunking, hierarchical chunking
4. **Query Expansion**: Generate multiple query variations
5. **Production Deployment**: Add caching, monitoring, analytics

### Production Considerations:
- **Chunk Deduplication**: Avoid storing duplicate chunks
- **Multi-tenancy**: Support multiple users with isolation
- **Caching**: Cache frequent queries and embeddings
- **Monitoring**: Track latency, accuracy, user satisfaction
- **Cost Optimization**: Balance quality vs. computational cost

### Congratulations! 🎉
You've built a production-ready RAG system!